In [1]:
# 一、准备文档
from langchain_text_splitters import MarkdownHeaderTextSplitter


def prepare_docs():
    # =============1.准备文档================
    with open('../day07/resources/评估.md', 'r', encoding='utf-8') as f:
        doc = f.read()

    # =============2.切分文档===============
    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        # 根据标题切分,然后会创建标题属性,这是langchain封装的向量数据库包会自动转成字段的依据
        [
        ("#", "Header1"),
        ("##", "HeaderH2"),
        ("###", "HeaderH3"),
    ],
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )

    # 切分文档, 返回的是 list[Document]
    docs = markdown_splitter.split_text(doc)

    # 为了方便查看，给每个Document加一个id，从1开始递增，但要注意的是LangChain的文档id要求是str
    for i, doc in enumerate(docs, start=1):
        doc.id = str(i)
    return docs

docs = prepare_docs()

In [2]:
"""
    创建向量化模型
"""
from langchain_ollama import OllamaEmbeddings
# 创建向量模型,我们今天使用ollama
ollama_embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b", dimensions=1024)

"""
    创建langchain包装过的向量数据库
"""
from app.core.config import settings
# 初始化向量数据库客户端
from langchain_milvus import Milvus, BM25BuiltInFunction

vector_store = Milvus(
    embedding_function=ollama_embeddings,  # 稠密向量模型
    collection_name="langchain_collection",  # collection名称
    builtin_function=BM25BuiltInFunction(  # 生成稀疏向量的函数
        analyzer_params={"type": "chinese"}  # 指定中文分词
    ),
    vector_field=["dense", "sparse"],  # 向量字段，包括稠密和稀疏
    connection_args={
        "uri": settings.rag.milvus_url,  # milvus的uri路径
    },
    drop_old=True,  # 是否删除旧的collection，避免重复创建
    # 自动主键
    auto_id=False
)

In [3]:
"""
    将文档片段写入向量数据库
"""
# 添加文档
batch_documents = [docs[i:i + 20] for i in range(0, len(docs), 20)]

for batch in batch_documents:
    # 显示传入id
    ids = [doc.id for doc in batch]  # 取出自定义 ID（字符串列表）
    result = vector_store.add_documents(batch,ids=ids)
    print(result)

['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20']
['21']


In [8]:
"""
    删除操作
"""
vector_store.delete(ids=['1'])

True